In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

In [5]:
dfdata = pd.read_csv("data_11_pima.csv")
dfdata.head()

,pregnant,glucose,pressure,triceps,insulin,mass,pedigree,age,outcome
0,7,155,56,28,81,46.5,0.526,47,1
1,1,93,78,14,59,35.1,0.597,45,0
2,0,88,76,47,53,30.9,0.325,25,0
3,2,109,76,34,48,35.2,0.692,23,0
4,0,187,64,23,190,26.4,0.514,22,1


In [6]:
X = dfdata.iloc[:, :-1]  # All columns except the last
y = dfdata.iloc[:, -1]   # Last column (outcome)

# Define dataset sizes to test
dataset_sizes = [100, 1000, 10000, 100000, 1000000, 10000000]
results = []

# Initialize XGBoost classifier
xgb_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

# Function to evaluate model for a given dataset size
def evaluate_model(X, y, size):
    # Ensure we don't try to use more samples than available
    size = min(size, len(X))

    # Sample data to the desired size
    if size < len(X):
        X_sample, _, y_sample, _ = train_test_split(
            X, y, train_size=size, random_state=42, stratify=y
        )
    else:
        X_sample, y_sample = X, y

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
    )

    # Measure training time
    start_time = time.time()

    # Perform 5-fold cross-validation
    cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy')

    # Train the model on the full training set
    xgb_model.fit(X_train, y_train)

    # Calculate total training time
    training_time = time.time() - start_time

    # Test set predictions
    y_pred = xgb_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    return {
        'Dataset Size': size,
        'CV Accuracy': cv_scores.mean(),
        'Test Accuracy': test_accuracy,
        'Training Time (s)': training_time
    }

# Evaluate for each dataset size
for size in dataset_sizes:
    # Skip sizes larger than our dataset
    if size > len(X):
        print(f"Skipping size {size} as it exceeds dataset length ({len(X)})")
        continue

    print(f"Evaluating model with dataset size: {size}")
    result = evaluate_model(X, y, size)
    results.append(result)

# Convert results to DataFrame for easy viewing
results_df = pd.DataFrame(results)
print("\nResults:")
print(results_df)


Evaluating model with dataset size: 100
Evaluating model with dataset size: 1000
Evaluating model with dataset size: 10000
Evaluating model with dataset size: 100000
Evaluating model with dataset size: 1000000
Evaluating model with dataset size: 10000000

Results:
   Dataset Size  CV Accuracy  Test Accuracy  Training Time (s)
0           100     0.912500       0.850000           0.266212
1          1000     0.928750       0.945000           0.404746
2         10000     0.969500       0.972000           1.631293
3        100000     0.982875       0.984150           6.325862
4       1000000     0.987086       0.987520          48.109797
5      10000000     0.988206       0.988244         442.848371
